# Low-dimensional Q70 quantile-alignment experiment (v1)

This notebook studies a low-dimensional conditional-independence DGP in which the conditional **70th percentile**, rather than the conditional mean or median, changes with $Z$.

It compares two **separately tuned** conditional-generator specifications:

1. **No alignment**: pure conditional MMD training.
2. **Q70 alignment**: conditional MMD plus a pinball-loss penalty at $\tau=0.70$.

The final H0 size experiment uses exactly **100 Monte Carlo repetitions** for each method at $n=400$. The selected alignment weight is **$\lambda=0.05$**, which was the best previous Q70 setting among the values already examined. The notebook also retains the H1 power experiment, using H0 p-values to report size-adjusted power.

> Important: the learning parameters below are a targeted, theory- and prior-result-based retuning. No parameter choice can guarantee nominal rejection rates before the 100-repetition H0 experiment is run. The notebook therefore checks an explicit Monte Carlo fluctuation band before treating power as interpretable.


## 1. Imports and project module

Place `ci_test.py` in the same directory as this notebook. The fallback search paths below also cover the supplied project layout and a typical Colab upload directory.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


def load_ci_test():
    try:
        import ci_test as module
        return module
    except ModuleNotFoundError:
        candidates = [
            Path.cwd() / "ci_test.py",
            Path.cwd() / "project_sources" / "10-ci_test.py",
            Path("/content/ci_test.py"),
        ]
        for path in candidates:
            if path.exists():
                spec = importlib.util.spec_from_file_location("ci_test", path)
                module = importlib.util.module_from_spec(spec)
                sys.modules["ci_test"] = module
                spec.loader.exec_module(module)
                return module
        raise FileNotFoundError(
            "Cannot find ci_test.py. Put it in the same directory as this notebook."
        )


C = load_ci_test()
DEVICE = C.get_device(prefer_gpu=True)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


## 2. Q70 DGP

For $h>0$, define a piecewise quantile function $Q_h(u)$ whose mean and median remain zero while its 70th percentile equals $h$. The functions $h_X(Z)$ and $h_Y(Z)$ change nonlinearly with a 10-dimensional Gaussian $Z$.

Under H0, $U_X$ and $U_Y$ are independent. Under H1, their marginal distributions remain Uniform$(0,1)$, but the two indicators $1(U_X>0.7)$ and $1(U_Y>0.7)$ are coupled through a checkerboard copula. Thus dependence is introduced specifically around Q70 without changing either conditional marginal.


In [ ]:
def _q70_levels(Z):
    if Z.ndim != 2 or Z.shape[1] < 5:
        raise ValueError("Q70 DGP requires Z with at least five columns.")
    hx = 0.8 + 0.7 * torch.sigmoid(
        0.8 * Z[:, [0]] - 0.6 * Z[:, [1]] + 0.4 * torch.sin(Z[:, [2]])
    )
    hy = 0.8 + 0.7 * torch.sigmoid(
        -0.7 * Z[:, [0]] + 0.5 * Z[:, [3]] + 0.3 * torch.cos(Z[:, [4]])
    )
    return hx, hy


def _q70_quantile(u, h):
    """Piecewise inverse CDF: mean=0, median=0, and Q(0.70)=h."""
    c = 16.0 * h - 8.8
    left_tail = -c + (u / 0.05) * (c - 1.0)
    center_left = -1.0 + (u - 0.05) / 0.45
    center_right = h * (u - 0.50) / 0.20
    upper_tail = h + 0.2 * (u - 0.70) / 0.30
    return torch.where(
        u < 0.05,
        left_tail,
        torch.where(u < 0.50, center_left, torch.where(u < 0.70, center_right, upper_tail)),
    )


def _checkerboard_uniforms(n, tau, rho, device):
    """Uniform marginals with dependence only in cells split at `tau`."""
    if not (0.0 <= rho <= 1.0):
        raise ValueError("alpha_x/rho must be between 0 and 1.")
    delta = rho * tau * (1.0 - tau)
    probs = torch.tensor(
        [
            tau * tau + delta,
            tau * (1.0 - tau) - delta,
            tau * (1.0 - tau) - delta,
            (1.0 - tau) ** 2 + delta,
        ],
        dtype=torch.float32,
        device=device,
    )
    cell = torch.multinomial(probs, n, replacement=True)
    x_high = (cell >= 2).reshape(-1, 1)
    y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)
    ux = torch.where(
        x_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )
    uy = torch.where(
        y_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )
    return ux, uy


def sample_q70(n, hypothesis="H0", device=None, dz=10, alpha_x=0.10, **_):
    device = device or C.get_device(prefer_gpu=True)
    if dz < 5:
        raise ValueError("Use dz >= 5 for this Q70 DGP.")
    Z = torch.randn(n, dz, device=device)
    hx, hy = _q70_levels(Z)

    if hypothesis.upper() == "H0":
        ux = torch.rand(n, 1, device=device)
        uy = torch.rand(n, 1, device=device)
    elif hypothesis.upper() == "H1":
        ux, uy = _checkerboard_uniforms(n, tau=0.70, rho=float(alpha_x), device=device)
    else:
        raise ValueError("hypothesis must be 'H0' or 'H1'.")

    X = _q70_quantile(ux, hx)
    Y = _q70_quantile(uy, hy)
    return X, Y, Z


def oracle_q70(Z, m, device=None, **_):
    """Independent draws from the true P(X|Z) and P(Y|Z)."""
    device = device or Z.device
    Z = Z.to(device)
    hx, hy = _q70_levels(Z)
    hx = hx.unsqueeze(1)  # (n, 1, 1), broadcast over m draws
    hy = hy.unsqueeze(1)
    ux = torch.rand(Z.shape[0], m, 1, device=device)
    uy = torch.rand(Z.shape[0], m, 1, device=device)
    return _q70_quantile(ux, hx), _q70_quantile(uy, hy)


C.register_dgp(C.DGP(
    name="q70_lowdim",
    sample=sample_q70,
    oracle=oracle_q70,
    description=(
        "Low-dimensional Q70 DGP: E[X|Z]=E[Y|Z]=0 and conditional medians are zero, "
        "while Q0.70 varies nonlinearly with Z. H1 couples the two Q70 exceedance indicators."
    ),
))

print(C.DGPS["q70_lowdim"].description)


### Quick DGP validation

The first check validates dimensions. The second approximates the conditional mean, median, and Q70 at a fixed $Z=0$; small Monte Carlo deviations are expected.


In [ ]:
torch.manual_seed(2026)
X0, Y0, Z0 = sample_q70(400, hypothesis="H0", device=DEVICE, dz=10)
X1, Y1, Z1 = sample_q70(400, hypothesis="H1", device=DEVICE, dz=10, alpha_x=0.20)
assert X0.shape == Y0.shape == (400, 1) and Z0.shape == (400, 10)
assert X1.shape == Y1.shape == (400, 1) and Z1.shape == (400, 10)

z_fixed = torch.zeros(1, 10, device=DEVICE)
xf, yf = oracle_q70(z_fixed, m=50_000, device=DEVICE)
hx0, hy0 = _q70_levels(z_fixed)
diagnostic = pd.DataFrame({
    "target": ["X", "Y"],
    "sample mean": [xf.mean().item(), yf.mean().item()],
    "sample median": [xf.median().item(), yf.median().item()],
    "sample Q70": [torch.quantile(xf, 0.70).item(), torch.quantile(yf, 0.70).item()],
    "theoretical Q70": [hx0.item(), hy0.item()],
})
display(diagnostic.round(4))


## 3. Fixed experiment settings and separately tuned networks

The sample size remains $n=400$, matching the preceding experiments. Every reported H0 or H1 point uses 100 Monte Carlo datasets.

The two generator specifications are tuned separately because the Q70 pinball term changes the scale and geometry of the training objective. Both use three hidden layers and a learning-rate scheduler, but the no-alignment model receives more generated MMD draws and a slightly faster initial learning rate, whereas the aligned model uses a more conservative rate, more alignment draws, and a longer patience window.

These choices address underfitting/early stopping without changing the DGP, sample size, number of repetitions, test statistic, or bootstrap size.


In [ ]:
# ----- experiment constants -----
N = 400
N_REP_H0 = 100
N_REP_H1 = 100
LEVELS = (0.10, 0.05)
ALPHA_GRID = (0.05, 0.10, 0.15, 0.20, 0.25)
DGP_NAME = "q70_lowdim"
BASE_DATA_KWARGS = dict(dz=10)

# With 100 repetitions these are practical Monte Carlo fluctuation bands around
# nominal 0.05 and 0.10. They are diagnostics, not acceptance guarantees.
NORMAL_BANDS = {
    0.05: (0.01, 0.10),
    0.10: (0.04, 0.16),
}

# ci_test.py automatically uses n_jobs=1 on a CUDA/MPS device.
N_JOBS = 1 if DEVICE.type in ("cuda", "mps") else -1

COMMON_CONFIG = dict(
    depth=3,
    width=1024,
    noise_dim=8,
    noise_var=1.0,
    noise_kind="normal",
    dropout=0.0,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,
    early_stop=True,
    lr_scheduler=True,
    lr_factor=0.30,
    lr_patience=20,
    min_lr_frac=0.02,
    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv="gaussian",
    standardize=True,
)

# Pure conditional MMD: a slightly faster start and more MMD draws stabilize the
# learned residual distribution in the absence of a quantile anchor.
CONFIG_NO_ALIGNMENT = {
    **COMMON_CONFIG,
    "lr": 1.0e-3,
    "epochs": 1000,
    "M_train": 40,
    "min_epochs": 140,
    "patience": 110,
    "min_delta": 2.0e-5,
    "align_mode": "none",
    "lambda_align": 0.0,
    "taus": (0.70,),
    "align_samples": 64,
}

# Q70 alignment: lambda=0.05 was the preferred value in the preceding Q70
# comparison. A lower learning rate and more alignment draws reduce noisy Q70 updates.
CONFIG_Q70_ALIGNMENT = {
    **COMMON_CONFIG,
    "lr": 7.5e-4,
    "epochs": 1100,
    "M_train": 30,
    "min_epochs": 160,
    "patience": 130,
    "min_delta": 2.0e-5,
    "align_mode": "quantile",
    "lambda_align": 0.05,
    "taus": (0.70,),
    "align_samples": 128,
}

METHODS = {
    "No alignment (lambda=0)": CONFIG_NO_ALIGNMENT,
    "Q70 alignment (lambda=0.05)": CONFIG_Q70_ALIGNMENT,
}

rows = []
for method, cfg in METHODS.items():
    rows.append({
        "method": method,
        "depth x width": f"{cfg['depth']} x {cfg['width']}",
        "learning rate": cfg["lr"],
        "epoch cap": cfg["epochs"],
        "M_train": cfg["M_train"],
        "min epochs": cfg["min_epochs"],
        "patience": cfg["patience"],
        "alignment mode": cfg["align_mode"],
        "tau": cfg["taus"],
        "lambda": cfg["lambda_align"],
        "alignment draws": cfg["align_samples"],
    })
display(pd.DataFrame(rows))


## 4. Oracle H0 sanity check — 100 repetitions

The oracle uses the known conditional marginals, so it diagnoses the test/bootstrap implementation independently of neural-network approximation. If the oracle is seriously non-nominal, stop and inspect the DGP or test implementation before tuning either network.


In [ ]:
t0 = time.time()
oracle_h0 = C.run_experiment(
    n=N,
    hypothesis="H0",
    n_rep=N_REP_H0,
    config=CONFIG_NO_ALIGNMENT,
    oracle=True,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    dgp=DGP_NAME,
    n_jobs=N_JOBS,
    prefer_gpu=True,
    verbose=True,
)
print(f"Oracle elapsed: {(time.time() - t0) / 60:.1f} minutes")


## 5. Learned-generator H0 size — 100 repetitions per method

Both methods use the same 100 data seeds, giving a paired comparison. This is the decisive check for whether the retuned networks produce rejection rates within ordinary Monte Carlo fluctuation.


In [ ]:
h0_results = {}

for method, cfg in METHODS.items():
    print("\n" + "=" * 88)
    print(method)
    print("=" * 88)
    t0 = time.time()
    h0_results[method] = C.run_experiment(
        n=N,
        hypothesis="H0",
        n_rep=N_REP_H0,
        config=cfg,
        oracle=False,
        levels=LEVELS,
        data_kwargs=BASE_DATA_KWARGS,
        dgp=DGP_NAME,
        n_jobs=N_JOBS,
        prefer_gpu=True,
        verbose=True,
    )
    print(f"Elapsed: {(time.time() - t0) / 60:.1f} minutes")

np.savez_compressed(
    "q70_h0_calibration_v1.npz",
    oracle_pvalues=oracle_h0["pvalues"],
    no_alignment_pvalues=h0_results["No alignment (lambda=0)"]["pvalues"],
    q70_alignment_pvalues=h0_results["Q70 alignment (lambda=0.05)"]["pvalues"],
)
print("Saved q70_h0_calibration_v1.npz")


### Size summary and diagnostic decision

With 100 repetitions, one rejection changes the reported rate by 0.01. We flag a method as acceptable when both rejection rates lie inside the stated Monte Carlo bands. The empirical H0 p-value quantiles are also saved as size-adjusted cutoffs for the subsequent power comparison.


In [ ]:
def empirical_cutoff(pvalues, level):
    # Linear interpolation is retained explicitly for reproducibility across NumPy versions.
    return float(np.quantile(np.asarray(pvalues), level, method="linear"))


size_rows = []
all_h0 = {"Oracle": oracle_h0, **h0_results}
for method, result in all_h0.items():
    p = np.asarray(result["pvalues"])
    row = {"method": method, "n_rep": len(p)}
    checks = []
    for level in LEVELS:
        rate = float(np.mean(p < level))
        low, high = NORMAL_BANDS[level]
        ok = low <= rate <= high
        checks.append(ok)
        row[f"rej@{level:.2f}"] = rate
        row[f"normal band@{level:.2f}"] = f"[{low:.2f}, {high:.2f}]"
        row[f"in band@{level:.2f}"] = ok
        row[f"H0 cutoff@{level:.2f}"] = empirical_cutoff(p, level)
    row["both levels in band"] = all(checks)
    size_rows.append(row)

size_summary = pd.DataFrame(size_rows)
display(size_summary)
size_summary.to_csv("q70_h0_summary_v1.csv", index=False)

learned_ok = bool(
    size_summary.loc[
        size_summary["method"].isin(METHODS), "both levels in band"
    ].all()
)

if learned_ok:
    print("PASS: both learned-generator specifications are within the diagnostic bands.")
else:
    print(
        "CAUTION: at least one learned-generator specification is outside a diagnostic band. "
        "Do not interpret its raw power as a clean method comparison. Inspect the direction "
        "of the size distortion and retune the learning schedule before the final report."
    )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, level in zip(axes, (0.05, 0.10)):
    plot_df = size_summary.set_index("method")
    vals = plot_df[f"rej@{level:.2f}"]
    ax.bar(vals.index, vals.values, color=["#8c8c8c", "#4c78a8", "#f58518"])
    low, high = NORMAL_BANDS[level]
    ax.axhline(level, color="black", linestyle="--", linewidth=1.2, label="nominal")
    ax.axhspan(low, high, color="green", alpha=0.12, label="MC band")
    ax.set_title(f"H0 rejection rate at {level:.2f}")
    ax.set_ylim(0, max(0.22, vals.max() + 0.04))
    ax.tick_params(axis="x", rotation=22)
    ax.legend()
plt.tight_layout()
plt.show()


## 6. H1 power experiment — retained for validation

For each dependence strength, the code reports:

- **Raw power**: rejection using the nominal p-value threshold.
- **Size-adjusted power**: rejection using that method's empirical H0 p-value quantile.

Size adjustment makes the comparison more informative when finite-sample H0 rejection is not exactly nominal. The H1 experiment also uses 100 repetitions per method and dependence level. The cell is enabled by default only if both learned specifications passed the H0 diagnostic; set `RUN_POWER=True` manually only if you deliberately want exploratory results despite an H0 warning.


In [ ]:
RUN_POWER = learned_ok

h0_cutoffs = {
    method: {
        level: empirical_cutoff(result["pvalues"], level)
        for level in LEVELS
    }
    for method, result in h0_results.items()
}

power_results = {}
power_rows = []

if not RUN_POWER:
    print(
        "Power was not run because at least one learned H0 size is outside the diagnostic band. "
        "After retuning and rerunning Sections 5-6, set RUN_POWER=True to run this cell."
    )
else:
    for alpha_x in ALPHA_GRID:
        for method, cfg in METHODS.items():
            print("\n" + "-" * 88)
            print(f"alpha_x={alpha_x:.2f} | {method}")
            result = C.run_experiment(
                n=N,
                hypothesis="H1",
                n_rep=N_REP_H1,
                config=cfg,
                oracle=False,
                levels=LEVELS,
                data_kwargs={**BASE_DATA_KWARGS, "alpha_x": alpha_x},
                dgp=DGP_NAME,
                n_jobs=N_JOBS,
                prefer_gpu=True,
                verbose=True,
            )
            power_results[(alpha_x, method)] = result
            p = np.asarray(result["pvalues"])
            for level in LEVELS:
                power_rows.append({
                    "alpha_x": alpha_x,
                    "method": method,
                    "level": level,
                    "raw_power": float(np.mean(p < level)),
                    "h0_adjusted_cutoff": h0_cutoffs[method][level],
                    "size_adjusted_power": float(np.mean(p < h0_cutoffs[method][level])),
                    "n_rep": len(p),
                })

    power_summary = pd.DataFrame(power_rows)
    display(power_summary)
    power_summary.to_csv("q70_power_size_adjusted_v1.csv", index=False)
    print("Saved q70_power_size_adjusted_v1.csv")


In [ ]:
if RUN_POWER and len(power_rows):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, level in zip(axes, (0.05, 0.10)):
        d = power_summary[power_summary["level"] == level]
        for method in METHODS:
            dm = d[d["method"] == method].sort_values("alpha_x")
            ax.plot(
                dm["alpha_x"], dm["size_adjusted_power"],
                marker="o", linewidth=2, label=method,
            )
        ax.axhline(level, color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"Size-adjusted power at level {level:.2f}")
        ax.set_xlabel("Q70 dependence strength (alpha_x)")
        ax.set_ylabel("rejection rate")
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()


## 7. Reporting checklist

For the final report, record only the following compact items:

1. Oracle H0 rejection at 0.05 and 0.10.
2. Each learned method's H0 rejection at 0.05 and 0.10, with the pass/warning diagnostic.
3. The chosen Q70 setting: $\tau=0.70$, $\lambda=0.05$.
4. Size-adjusted power curves (or a short table) across `ALPHA_GRID`.
5. State clearly that the two networks were tuned separately, so this is a performance comparison rather than a strict one-factor ablation.

If H0 is too liberal, the first conservative adjustment is to lower that method's learning rate by 25% and increase `min_epochs`/`patience` by 20; if it is too conservative, reverse those changes modestly. Keep $n=400$, 100 repetitions, the DGP, bootstrap size, and $\lambda=0.05$ fixed while retuning.
